# Dippy Animation Trajectory Generator

This notebook launches the Dippy app — a Gradio interface for chained WAN 2.1 Image-to-Video generation with frame continuity.

**Requirements:** A Colab runtime with GPU (T4 or better). An HF token and optionally an OpenAI API key configured in Colab Secrets.

In [ ]:
# Cell 1: Mount Google Drive (for model caching)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2: Clone the dippy-WAN repo
import os
if not os.path.exists('/content/dippy-WAN'):
    # Option A: Clone from GitHub (update URL to your repo)
    # !git clone https://github.com/YOUR_USER/dippy-WAN.git /content/dippy-WAN

    # Option B: If you uploaded the files directly, skip this cell
    print('dippy-WAN directory not found.')
    print('Please either:')
    print('  1. Update the git clone URL above and uncomment it, or')
    print('  2. Upload dippy-app.py to /content/dippy-WAN/')
else:
    print('dippy-WAN directory found.')

In [ ]:
# Cell 3: Install pinned dependencies (WAN-compatible)
!pip install -q --upgrade \
  "git+https://github.com/huggingface/diffusers.git@3a23d941f559759195dd30b5d206008f9e34f2bb" \
  "transformers==4.55.4" \
  "accelerate==1.10.0" \
  "huggingface_hub==0.34.4" \
  gradio spaces ftfy peft imageio-ffmpeg opencv-python safetensors sentencepiece openai

# Print resolved versions so mismatches are obvious
import importlib.metadata as md
for pkg in ("diffusers", "transformers", "accelerate", "huggingface_hub"):
    try:
        print(f"{pkg}=={md.version(pkg)}")
    except Exception:
        print(f"{pkg} not installed")


In [ ]:
# Cell 4: Set API keys and cache directory
import os
from google.colab import userdata

# Point HF downloads to Google Drive cache (must be set before importing HF libs)
CACHE_DIR = '/content/drive/My Drive/huggingface_cache'
os.makedirs(CACHE_DIR, exist_ok=True)
os.environ['HF_HOME'] = CACHE_DIR
os.environ['HF_HUB_CACHE'] = CACHE_DIR
os.environ['HF_ASSETS_CACHE'] = os.path.join(CACHE_DIR, 'assets')
os.environ['HF_XET_CACHE'] = os.path.join(CACHE_DIR, 'xet')
os.makedirs(os.environ['HF_ASSETS_CACHE'], exist_ok=True)
os.makedirs(os.environ['HF_XET_CACHE'], exist_ok=True)

print(f"HF_HOME: {os.environ['HF_HOME']}")
print(f"HF_HUB_CACHE: {os.environ['HF_HUB_CACHE']}")
print(f"HF_ASSETS_CACHE: {os.environ['HF_ASSETS_CACHE']}")
print(f"HF_XET_CACHE: {os.environ['HF_XET_CACHE']}")

# Hugging Face token (required for model download)
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN loaded successfully.')
except Exception as e:
    print(f'Could not load HF_TOKEN: {e}')
    print('Please add a secret named HF_TOKEN in the Colab secrets tab (key icon).')

# OpenAI API key (optional, for LLM sentence generation)
try:
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    print('OPENAI_API_KEY loaded successfully.')
except Exception as e:
    print(f'Could not load OPENAI_API_KEY: {e}')
    print('Sentence generation via LLM will not be available.')
    print('You can still type sentences manually.')


In [ ]:
# Cell 5: Launch the Dippy app
!python /content/dippy-WAN/dippy-app.py